In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[4]") \
    .appName("RDD Assignment") \
    .getOrCreate()

### Task 1


In [10]:
rdd = spark.sparkContext.textFile("file:///data/employees.txt")

rdd.collect()

['emp_id,name,department,job_title,salary,location,hire_date,performance_rating,years_exp',
 '1,John Smith,Engineering,Senior Developer,125000,San Francisco,2021-03-15,4.5,8',
 '2,Sarah Johnson,Sales,Account Executive,85000,New York,2022-01-10,4.2,5',
 '3,Michael Williams,Engineering,Software Engineer,95000,Austin,2023-06-20,3.8,3',
 '4,Jennifer Brown,Marketing,Marketing Manager,92000,Chicago,2020-11-05,4.7,7',
 '',
 '5,David Jones,Finance,Senior Analyst,105000,Boston,2021-08-12,4.3,6',
 '6,Lisa Garcia,IT,DevOps Engineer,115000,Seattle,2022-04-18,4.6,4',
 '7,Robert Martinez,Legal,Legal Counsel145000,San Francisco,2019-09-22,4.8,10, 5',
 '8,Patricia Wilson,HR,HR Manager,88000,Denver,2020-02-28,4.1,6',
 '',
 '9,James Anderson,Sales,Sales Manager,110000,Atlanta,2021-12-01,4.4,7',
 '10,Mary Thomas,Engineering,Tech Lead,145000,Seattle,2018-07-14,4.9,12']

In [16]:
header = rdd.first()

data_rdd = rdd.filter(lambda line: line != header)

parsed_rdd = data_rdd.map(lambda line: line.split(","))
parsed_rdd.collect()

[['1',
  'John Smith',
  'Engineering',
  'Senior Developer',
  '125000',
  'San Francisco',
  '2021-03-15',
  '4.5',
  '8'],
 ['2',
  'Sarah Johnson',
  'Sales',
  'Account Executive',
  '85000',
  'New York',
  '2022-01-10',
  '4.2',
  '5'],
 ['3',
  'Michael Williams',
  'Engineering',
  'Software Engineer',
  '95000',
  'Austin',
  '2023-06-20',
  '3.8',
  '3'],
 ['4',
  'Jennifer Brown',
  'Marketing',
  'Marketing Manager',
  '92000',
  'Chicago',
  '2020-11-05',
  '4.7',
  '7'],
 [''],
 ['5',
  'David Jones',
  'Finance',
  'Senior Analyst',
  '105000',
  'Boston',
  '2021-08-12',
  '4.3',
  '6'],
 ['6',
  'Lisa Garcia',
  'IT',
  'DevOps Engineer',
  '115000',
  'Seattle',
  '2022-04-18',
  '4.6',
  '4'],
 ['7',
  'Robert Martinez',
  'Legal',
  'Legal Counsel145000',
  'San Francisco',
  '2019-09-22',
  '4.8',
  '10',
  ' 5'],
 ['8',
  'Patricia Wilson',
  'HR',
  'HR Manager',
  '88000',
  'Denver',
  '2020-02-28',
  '4.1',
  '6'],
 [''],
 ['9',
  'James Anderson',
  'Sales',

### Task 2


In [15]:
name_count = (
    parsed_rdd
    .filter(lambda x: len(x) == 9)
    .map(lambda x: (x[1], 1))
    .reduceByKey(lambda a, b: a + b)
)

name_count.collect()

[('Robert Martinez', 1),
 ('Patricia Wilson', 1),
 ('James Anderson', 1),
 ('Mary Thomas', 1),
 ('Sarah Johnson', 1),
 ('Michael Williams', 1),
 ('Jennifer Brown', 1),
 ('David Jones', 1),
 ('John Smith', 1),
 ('Lisa Garcia', 1)]

### Task 3

In [17]:
valid_rdd = parsed_rdd.filter(lambda x: len(x) == 9)

valid_rdd.take(5)

[['1',
  'John Smith',
  'Engineering',
  'Senior Developer',
  '125000',
  'San Francisco',
  '2021-03-15',
  '4.5',
  '8'],
 ['2',
  'Sarah Johnson',
  'Sales',
  'Account Executive',
  '85000',
  'New York',
  '2022-01-10',
  '4.2',
  '5'],
 ['3',
  'Michael Williams',
  'Engineering',
  'Software Engineer',
  '95000',
  'Austin',
  '2023-06-20',
  '3.8',
  '3'],
 ['4',
  'Jennifer Brown',
  'Marketing',
  'Marketing Manager',
  '92000',
  'Chicago',
  '2020-11-05',
  '4.7',
  '7'],
 ['5',
  'David Jones',
  'Finance',
  'Senior Analyst',
  '105000',
  'Boston',
  '2021-08-12',
  '4.3',
  '6']]

### Task 4

In [22]:
def safe_convert(x):
    try:
        return (x[2], (float(x[4]), 1))
    except (ValueError, IndexError):
        return None  # Skip invalid rows

avg_salary = (
    valid_rdd
    .map(safe_convert)
    .filter(lambda x: x is not None)  # Remove None values
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
    .mapValues(lambda x: x[0] / x[1])
)
avg_salary.collect()

[('Sales', 97500.0),
 ('Finance', 105000.0),
 ('Engineering', 121666.66666666667),
 ('Marketing', 92000.0),
 ('IT', 115000.0),
 ('HR', 88000.0)]

### Task 5

In [24]:
dept_count = (
    valid_rdd
    .map(lambda x: (x[2], 1))
    .reduceByKey(lambda a, b: a + b)
)

dept_count.collect()

[('Legal', 1),
 ('Sales', 2),
 ('Finance', 1),
 ('Engineering', 3),
 ('Marketing', 1),
 ('IT', 1),
 ('HR', 1)]